In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
from src.data.load_data import load_datasets

datasets = load_datasets()

Loaded 9 datasets successfully.


In [4]:
orders_df = datasets["olist_orders_dataset"].copy()

In [5]:
orders_df.dtypes

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

In [6]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

In [7]:
import pandas as pd

In [8]:
for column in date_columns:

    orders_df[column] = pd.to_datetime(
        orders_df[column],
        errors="coerce"
    )

In [9]:
orders_df.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

## Datetime Conversion

### Objective
Convert timestamp columns from `object` to `datetime` for time-based analysis.

### Findings
- Successfully converted all five timestamp columns.
- Invalid dates, if any, were converted to `NaT`.
- The dataset is now ready for time-series feature engineering.

In [10]:
from src.utils.data_profiler import (
    profile_summary,
    missing_value_report
)

In [11]:
missing_report = missing_value_report(orders_df)

display(missing_report)

,Column,Missing Values,Missing %,Data Type
6,order_delivered_customer_date,2965,2.98,datetime64[ns]
5,order_delivered_carrier_date,1783,1.79,datetime64[ns]
4,order_approved_at,160,0.16,datetime64[ns]
0,order_id,0,0.00,object
3,order_purchase_timestamp,0,0.00,datetime64[ns]
2,order_status,0,0.00,object
1,customer_id,0,0.00,object
7,order_estimated_delivery_date,0,0.00,datetime64[ns]


In [12]:
orders_df["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [13]:
orders_df.loc[
    orders_df["order_delivered_customer_date"].isna(),
    "order_status"
].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [14]:
orders_df.loc[
    orders_df["order_delivered_carrier_date"].isna(),
    "order_status"
].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [15]:
orders_df.loc[
    orders_df["order_approved_at"].isna(),
    "order_status"
].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [16]:
orders_df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

## Missing Value Handling

### Objective
Investigate missing values and determine whether they represent data quality issues or valid business events.

### Decision
- Missing delivery timestamps were retained because they correspond to cancelled or unavailable orders.
- Missing approval timestamps were also retained for the same reason.
- No imputation was performed since filling these values would introduce incorrect business information.

### Outcome
The dataset preserves its business meaning while remaining suitable for downstream analysis.

In [17]:
duplicate_rows = orders_df.duplicated().sum()

print(f"Duplicate Rows: {duplicate_rows}")

Duplicate Rows: 0


In [18]:
orders_df["order_id"].nunique(), len(orders_df)

(99441, 99441)

In [19]:
order_items_df = datasets["olist_order_items_dataset"].copy()

In [20]:
order_items_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [21]:
profile_dataset(order_items_df, "Order Items Dataset")

NameError: name 'profile_dataset' is not defined